In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from datetime import date
from common import FeedID, Operator

operator = Operator.SL
myDate = date(2024, 6, 15)
hour = 10
feedId = FeedID.TripUpdates

In [ ]:
from koda import download_koda_rt_file, download_koda_static_file

KODA_API_KEY = os.getenv("KODA_API_KEY")
if not KODA_API_KEY:
    raise ValueError("KODA_API_KEY not found in environment variables")

download_koda_rt_file(operator, feedId, myDate, api_key=KODA_API_KEY, data_dir="../data/koda-rt")

download_koda_static_file(operator, myDate, api_key=KODA_API_KEY, data_dir="../data/koda-static")

File ../data/koda-rt/sl_2024-06-15.7z already exists. Skipping download.
File ../data/koda-static/sl_2024-06-15.zip already exists. Skipping download.


In [ ]:
from gtfs import  load_pb_file

data_folder = f"../data/koda-rt/data-tmp/sl/{feedId.value}/{myDate.year:04d}/{myDate.month:02d}/{myDate.day:02d}/{hour:02d}"

files_in_data_folder = os.listdir(data_folder)
print("Files in data folder:", files_in_data_folder)

first_file_path = os.path.join(data_folder, files_in_data_folder[0])

gtfs_feed_message = load_pb_file(first_file_path)

Files in data folder: ['sl-tripupdates-2024-06-15T10-00-03Z.pb', 'sl-tripupdates-2024-06-15T10-00-17Z.pb', 'sl-tripupdates-2024-06-15T10-00-31Z.pb', 'sl-tripupdates-2024-06-15T10-00-45Z.pb', 'sl-tripupdates-2024-06-15T10-01-13Z.pb', 'sl-tripupdates-2024-06-15T10-01-27Z.pb', 'sl-tripupdates-2024-06-15T10-01-41Z.pb', 'sl-tripupdates-2024-06-15T10-01-55Z.pb', 'sl-tripupdates-2024-06-15T10-02-09Z.pb', 'sl-tripupdates-2024-06-15T10-02-23Z.pb', 'sl-tripupdates-2024-06-15T10-02-37Z.pb', 'sl-tripupdates-2024-06-15T10-02-51Z.pb', 'sl-tripupdates-2024-06-15T10-03-05Z.pb', 'sl-tripupdates-2024-06-15T10-03-19Z.pb', 'sl-tripupdates-2024-06-15T10-03-33Z.pb', 'sl-tripupdates-2024-06-15T10-03-47Z.pb', 'sl-tripupdates-2024-06-15T10-04-01Z.pb', 'sl-tripupdates-2024-06-15T10-04-15Z.pb', 'sl-tripupdates-2024-06-15T10-04-29Z.pb', 'sl-tripupdates-2024-06-15T10-04-43Z.pb', 'sl-tripupdates-2024-06-15T10-05-11Z.pb', 'sl-tripupdates-2024-06-15T10-05-25Z.pb', 'sl-tripupdates-2024-06-15T10-05-39Z.pb', 'sl-tripupd

In [6]:
from __generated__ import gtfs_realtime_pb2


def debug_print_feed(entity: gtfs_realtime_pb2.FeedEntity):
    id = entity.id
    trip_update = entity.trip_update
    trip = trip_update.trip
    stop_time_update = trip_update.stop_time_update

    # print("Trip update")
    # print(trip_update)
    # print("Trip")
    # print(trip)
    # print("Stop time updates")
    # print(stop_time_update)

    # print(f"Entity ID: {entity.id}")
    for (id, a) in entity.trip_update.ListFields():
        print(f"  Field: {id.name}")
        # print(f"    Value: {a}")
    


# print(f"Parsed FeedMessage with {len(gtfs_data.entity)} entities")
gtfs_entity = gtfs_feed_message.entity[0]

debug_print_feed(gtfs_entity)

print("Total entities in feed:", len(gtfs_feed_message.entity))


  Field: trip
  Field: stop_time_update
  Field: vehicle
  Field: timestamp
Total entities in feed: 689


In [7]:
import pickle
from static import StaticData

static_data: StaticData
pickle_file_path = f"../data/koda-static/static-data-{myDate.year:04d}-{myDate.month:02d}-{myDate.day:02d}.pkl"
if os.path.exists(pickle_file_path):
    print("Static data pickle file exists.")

    static_data = pickle.load(open(pickle_file_path, "rb"))
else:
    print("Loading static data from GTFS files.")
    static_data = StaticData.load_static_data("../data/koda-static/data-tmp", myDate)
    pickle.dump(static_data, open(pickle_file_path, "wb"))
static_data

Static data pickle file exists.


In [8]:
from data_processing import feed_message_to_dataframe


df = feed_message_to_dataframe(gtfs_feed_message)

print(df.dtypes)

df.head()

id                                Int64
trip_id                  string[python]
start_date                       object
schedule_relationship             int64
vehicle_id                        Int64
stop_time_updates                object
timestamp                         int64
dtype: object


,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp
0,14010516247726276,14010000663741929,20240615,0,9031008000500536,"[{'stop_sequence': 25, 'stop_id': '90220010001...",1718438398
1,14010516247729020,14010000663747674,20240615,0,9031008000500539,"[{'stop_sequence': 11, 'stop_id': '90220010001...",1718438398
2,14010516089523031,14010000656705623,20240615,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398
3,14010516113220145,14010000656788822,20240615,0,9031001004302521,"[{'stop_sequence': 21, 'stop_id': '90220010061...",1718438398
4,14010516089526040,14010000656705875,20240615,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398


In [9]:
from data_processing import join_static_data_on_rt


df = join_static_data_on_rt(static_data, df)

In [10]:
df[(df["route_short_name"] == "40") & (df["route_desc"] == "Pendeltåg")].head()

,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
2,14010516089523031,14010000656705623,20240615,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg
4,14010516089526040,14010000656705875,20240615,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
19,14010516089527825,14010000656705974,20240615,0,9031001004002222,"[{'stop_sequence': 17, 'stop_id': '90220010050...",1718438398,9011001004000000,433,<NA>,0.0,4014010000492969316,14010000000001001,40,<NA>,100,Pendeltåg
25,14010516089529610,14010000656706199,20240615,0,9031001004002223,"[{'stop_sequence': 12, 'stop_id': '90220010053...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
113,14010516089531701,14010000656706391,20240615,0,9031001004002224,"[{'stop_sequence': 8, 'stop_id': '902200100516...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg


In [11]:
from data_processing import explode_to_stops_with_join_static

df_exploded_with_stop_times = explode_to_stops_with_join_static(static_data, df)

print(df_exploded_with_stop_times.dtypes)

df_exploded_with_stop_times.head()

id                                           Int64
start_date                                  object
schedule_relationship                        int64
vehicle_id                                   Int64
timestamp                                    int64
route_id                                    object
service_id                                   Int64
trip_headsign                       string[python]
direction_id                               float64
shape_id                                     Int64
agency_id                           string[python]
route_short_name                    string[python]
route_long_name                     string[python]
route_type                                   Int64
route_desc                          string[python]
stop_id                             string[python]
arrival_time                         datetime64[s]
departure_time                       datetime64[s]
stop_time_schedule_relationship              int64
arrival_time_planned           

id start_date  \
trip_id           stop_sequence                                 
14010000663741929 25             14010516247726276   20240615   
                  26             14010516247726276   20240615   
                  27             14010516247726276   20240615   
                  28             14010516247726276   20240615   
                  29             14010516247726276   20240615   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000663741929 25                                 0  9031008000500536   
                  26                                 0  9031008000500536   
                  27                                 0  9031008000500536   
                  28                                 0  9031008000500536   
                  29                                 0  9031008000500536   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000663741929 25             1718438398  9011008001300000           6   
                  26             1718438398  9011008001300000           6   
                  27             1718438398  9011008001300000           6   
                  28             1718438398  9011008001300000           6   
                  29             1718438398  9011008001300000           6   

                                trip_headsign  direction_id  \
trip_id           stop_sequence                               
14010000663741929 25                     <NA>           0.0   
                  26                     <NA>           0.0   
                  27                     <NA>           0.0   
                  28                     <NA>           0.0   
                  29                     <NA>           0.0   

                                            shape_id          agency_id  \
trip_id           stop_sequence                                           
14010000663741929 25             6014010000657796244  14010000000002071   
                  26             6014010000657796244  14010000000002071   
                  27             6014010000657796244  14010000000002071   
                  28             6014010000657796244  14010000000002071   
                  29             6014010000657796244  14010000000002071   

                                route_short_name route_long_name  route_type  \
trip_id           stop_sequence                                                
14010000663741929 25                          13            <NA>        1000   
                  26                          13            <NA>        1000   
                  27                          13            <NA>        1000   
                  28                          13            <NA>        1000   
                  29                          13            <NA>        1000   

                                      route_desc           stop_id  \
trip_id           stop_sequence                                      
14010000663741929 25             Waxholmsbolaget  9022001000143001   
                  26             Waxholmsbolaget  9022001000142001   
                  27             Waxholmsbolaget  9022001000141001   
                  28             Waxholmsbolaget  9022001000139001   
                  29             Waxholmsbolaget  9022001000138001   

                                       arrival_time      departure_time  \
trip_id           stop_sequence                                           
14010000663741929 25            2024-06-15 07:51:16 2024-06-15 07:52:13   
                  26            2024-06-15 07:53:16 2024-06-15 07:53:16   
                  27            2024-06-15 07:55:06 2024-06-15 07:55:49   
                  28            2024-06-15 07:59:29 2024-06-15 07:59:29   
                  29            2024-06-15 07:59:

In [12]:
# df_exploded_with_stop_times["arrival_time_late"] = (df_exploded_with_stop_times["arrival_time"] - df_exploded_with_stop_times["arrival_time_planned"])
# df_exploded_with_stop_times["departure_time_late"] = (df_exploded_with_stop_times["departure_time"] - df_exploded_with_stop_times["departure_time_planned"])

print(df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].dtypes)

df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].head()

arrival_time                datetime64[s]
arrival_time_planned       datetime64[ns]
arrival_time_late         timedelta64[ns]
departure_time              datetime64[s]
departure_time_planned     datetime64[ns]
departure_time_late       timedelta64[ns]
agency_id                  string[python]
dtype: object


arrival_time arrival_time_planned  \
trip_id           stop_sequence                                            
14010000663741929 25            2024-06-15 07:51:16  2024-06-15 07:46:00   
                  26            2024-06-15 07:53:16  2024-06-15 07:47:00   
                  27            2024-06-15 07:55:06  2024-06-15 07:50:00   
                  28            2024-06-15 07:59:29  2024-06-15 07:53:00   
                  29            2024-06-15 07:59:59  2024-06-15 07:54:00   

                                arrival_time_late      departure_time  \
trip_id           stop_sequence                                         
14010000663741929 25              0 days 00:05:16 2024-06-15 07:52:13   
                  26              0 days 00:06:16 2024-06-15 07:53:16   
                  27              0 days 00:05:06 2024-06-15 07:55:49   
                  28              0 days 00:06:29 2024-06-15 07:59:29   
                  29              0 days 00:05:59 2024-06-15 08:00:01   

                                departure_time_planned departure_time_late  \
trip_id           stop_sequence                                              
14010000663741929 25               2024-06-15 07:46:00     0 days 00:06:13   
                  26               2024-06-15 07:47:00     0 days 00:06:16   
                  27               2024-06-15 07:50:00     0 days 00:05:49   
                  28               2024-06-15 07:53:00     0 days 00:06:29   
                  29               2024-06-15 07:54:00     0 days 00:06:01   

                                         agency_id  
trip_id           stop_sequence                     
14010000663741929 25             14010000000002071  
                  26             14010000000002071  
                  27             14010000000002071  
                  28             14010000000002071  
                  29             14010000000002071

In [13]:
agencies = StaticData.load_agencies("../data/koda-static/data-tmp/agency.txt")

agencies.head()

,agency_name,agency_url,agency_timezone,agency_lang,agency_fare_url
agency_id,,,,,
14010000000001917,Upplands lokaltrafik,https://www.resrobot.se/,Europe/Stockholm,sv,<NA>
14010000000001983,Kommuner i Stockholms Län,https://www.resrobot.se/,Europe/Stockholm,sv,<NA>
14010000000001939,Länstrafiken Sörmland,https://www.resrobot.se/,Europe/Stockholm,sv,<NA>
14010000000002005,Färdtjänstnämnden,https://www.resrobot.se/,Europe/Stockholm,sv,<NA>
14010000000002071,Waxholmsbolaget Ångfartygs AB,https://www.resrobot.se/,Europe/Stockholm,sv,<NA>


In [14]:
SL_AGENCY_ID = agencies[agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]
WAXHOLMSBOLAGET_ID = agencies[agencies["agency_name"] == "Waxholmsbolaget Ångfartygs AB"].index[0]

route_dt = df_exploded_with_stop_times[
  (df_exploded_with_stop_times["route_short_name"] == "13") & \
  (df_exploded_with_stop_times["agency_id"] == WAXHOLMSBOLAGET_ID)]
route_dt.head(20)

id start_date  \
trip_id           stop_sequence                                 
14010000663741929 25             14010516247726276   20240615   
                  26             14010516247726276   20240615   
                  27             14010516247726276   20240615   
                  28             14010516247726276   20240615   
                  29             14010516247726276   20240615   
                  30             14010516247726276   20240615   
                  31             14010516247726276   20240615   
                  32             14010516247726276   20240615   
                  33             14010516247726276   20240615   
                  34             14010516247726276   20240615   
                  35             14010516247726276   20240615   
                  36             14010516247726276   20240615   
                  37             14010516247726276   20240615   
                  38             14010516247726276   20240615   
                  39             14010516247726276   20240615   
                  40             14010516247726276   20240615   
                  41             14010516247726276   20240615   
                  42             14010516247726276   20240615   
                  43             14010516247726276   20240615   
                  44             14010516247726276   20240615   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000663741929 25                                 0  9031008000500536   
                  26                                 0  9031008000500536   
                  27                                 0  9031008000500536   
                  28                                 0  9031008000500536   
                  29                                 0  9031008000500536   
                  30                                 0  9031008000500536   
                  31                                 0  9031008000500536   
                  32                                 0  9031008000500536   
                  33                                 0  9031008000500536   
                  34                                 0  9031008000500536   
                  35                                 0  9031008000500536   
                  36                                 0  9031008000500536   
                  37                                 0  9031008000500536   
                  38                                 0  9031008000500536   
                  39                                 0  9031008000500536   
                  40                                 0  9031008000500536   
                  41                                 0  9031008000500536   
                  42                                 0  9031008000500536   
                  43                                 0  9031008000500536   
                  44                                 0  9031008000500536   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000663741929 25             1718438398  9011008001300000           6   
                  26             1718438398  9011008001300000           6   
                  27             1718438398  9011008001300000           6   
                  28             1718438398  9011008001300000           6   
                  29             1718438398  9011008001300000           6   
                  30             1718438398  9011008001300000           6   
                  31             1718438398  9011008001300000           6   
                  32             1718438398  9011008001300000           6   
                  33             1718438398  9011008001300000           6   
                  34             1718438398  9011008001300000           6   
                  35

In [15]:
from gtfs import download_gtfs_static_file

GTFS_REGIONAL_STATIC_API_KEY = os.getenv("GTFS_REGIONAL_STATIC_API_KEY")
if not GTFS_REGIONAL_STATIC_API_KEY:
    raise ValueError("GTFS_REGIONAL_STATIC_API_KEY not found in environment variables")

download_gtfs_static_file(Operator.SL, api_key=GTFS_REGIONAL_STATIC_API_KEY, data_dir="../data/gtfs-static")

File ../data/gtfs-static/sl_gtfs_static.zip already exists. Skipping download.


In [16]:
from gtfs import download_gtfs_rt_file

GTFS_REGIONAL_RT_API_KEY = os.getenv("GTFS_REGIONAL_RT_API_KEY")
if not GTFS_REGIONAL_RT_API_KEY:
    raise ValueError("GTFS_REGIONAL_RT_API_KEY not found in environment variables")

download_gtfs_rt_file(Operator.SL, FeedID.TripUpdates, api_key=GTFS_REGIONAL_RT_API_KEY, data_dir="../data/gtfs-rt")

File ../data/gtfs-rt/sl_TripUpdates.pb already exists. Skipping download.
